In [1]:
import sys
sys.path.append("/mnt/digphat/syntheticDataBenchmark/Chuong_pipeline/")
import pandas as pd
import numpy as np
from SynOmics.processing.preprocessing import DataProcessor
from SynOmics.processing.metadata import MetaData

df = pd.read_csv("../../Data/Data_Braun/metadata.csv").drop(columns=["RNA_ID"])


In [31]:
df_attributes = df.iloc[:,1:]
df_attributes.tail()

,Cohort,Arm,MSKCC,Sarc,Rhab,Number_of_Prior_Therapies,Tumor_Sample_Primary_or_Metastasis,Sex,Age,ORR,Benefit,ExtremeResponder,PFS,PFS_CNSR,OS,OS_CNSR
1001,CM-009,NIVOLUMAB,NaN,NaN,NaN,NaN,METASTASIS,M,54.0,PD,NCB,PD,1.183562,1,16.405479,1
1002,CM-009,NIVOLUMAB,NaN,NaN,NaN,NaN,METASTASIS,M,64.0,PR,CB,NEITHER,16.438356,1,22.487671,1
1003,CM-009,NIVOLUMAB,NaN,NaN,NaN,NaN,METASTASIS,F,75.0,PD,NCB,PD,1.216438,1,19.134247,0
1004,CM-009,NIVOLUMAB,NaN,NaN,NaN,NaN,METASTASIS,M,54.0,PD,NCB,PD,1.216438,1,28.635616,0
1005,CM-009,NIVOLUMAB,NaN,NaN,NaN,NaN,METASTASIS,M,50.0,PD,NCB,PD,1.347945,1,1.347945,1


In [3]:
ordinal_cols = ["MSKCC", "Number_of_Prior_Therapies", "ORR","ExtremeResponder","Benefit"]
threshold_unique_values = 10
metadata = MetaData.get_metadata(
    data = df_attributes, 
    threshold_unique_values = threshold_unique_values,
    ordinal_features = ordinal_cols    
)

grouped_metadata = MetaData.grouping_features_astype(df_attributes, metadata)

In [4]:
encoded_ordinal_df, encoder = DataProcessor.encode_ordinal_features(
    data = df_attributes[grouped_metadata.get("ordinal_categorical")],
)

encoded_dummy_df = DataProcessor.encode_dummy_features(
    data = df_attributes[grouped_metadata.get("dummy_categorical")],
)

numerical_df, scaler_obj = DataProcessor.standardization(
    data = df_attributes[grouped_metadata.get("numerical")],
    scaler = "minmax"
)

In [5]:
processed_df = pd.concat([encoded_ordinal_df, encoded_dummy_df, numerical_df], axis = 1)

In [121]:
obj_lst = ["Cohort","Arm", "MSKCC", "Tumor_Sample_Primary_or_Metastasis","ORR","Benefit","Sex","ExtremeResponder"]
for col in obj_lst:
    df_attributes[col] = df_attributes[col].astype("category")

In [46]:
from sklearn.impute import MissingIndicator
import pandas as pd

mi = MissingIndicator(features='missing-only', error_on_new=False)
mask = mi.fit_transform(df_attributes)  # shape: (n_samples, n_missing_cols)
missing_cols = df_attributes.columns[mi.features_]  # Only missing-value columns
ind_cols = [f"missingindicator_{c}" for c in missing_cols]
indicators_df = pd.DataFrame(mask.astype(np.int), columns=ind_cols, index=df_attributes.index)

In [102]:
nan_dummy_features

['Cohort_nan',
 'Arm_nan',
 'Sarc_nan',
 'Rhab_nan',
 'Tumor_Sample_Primary_or_Metastasis_nan',
 'Sex_nan',
 'PFS_CNSR_nan',
 'OS_CNSR_nan']

In [99]:
dummy_cat_features_dict

{'Cohort': ['Cohort_CM-009', 'Cohort_CM-010', 'Cohort_CM-025', 'Cohort_nan'],
 'Arm': ['Arm_EVEROLIMUS', 'Arm_NIVOLUMAB', 'Arm_nan'],
 'Sarc': ['Sarc_0.0', 'Sarc_1.0', 'Sarc_nan'],
 'Rhab': ['Rhab_0.0', 'Rhab_1.0', 'Rhab_nan'],
 'Tumor_Sample_Primary_or_Metastasis': ['Tumor_Sample_Primary_or_Metastasis_METASTASIS',
  'Tumor_Sample_Primary_or_Metastasis_PRIMARY',
  'Tumor_Sample_Primary_or_Metastasis_nan'],
 'Sex': ['Sex_F', 'Sex_M', 'Sex_nan'],
 'PFS_CNSR': ['PFS_CNSR_0.0', 'PFS_CNSR_1.0', 'PFS_CNSR_nan'],
 'OS_CNSR': ['OS_CNSR_0.0', 'OS_CNSR_1.0', 'OS_CNSR_nan']}

In [ ]:
import miceforest as mf

# df_encoded: your preprocessed numeric DataFrame (incl. one-hot/ordinal encodings)
kernel = mf.ImputationKernel(
    data=data,
    num_datasets=1,                 # single completed dataset
    random_state=42
)

# Fit MICE with RandomForest-like (LightGBM) models
kernel.mice(
    iterations=10,              # increase for harder problems
    n_estimators=300,           # trees per model
    verbose=True
)

# Get completed data
df_imputed_mice = kernel.complete_data(dataset=0)

In [2]:
import pandas as pd
import numpy as np

np.random.seed(42)  # for reproducibility

# Step 1: create full dataset
n = 100
sex_choices = ["M", "F"]
stage_choices = ["low", "medium", "high"]

bigger_df = pd.DataFrame({
    "sex": np.random.choice(sex_choices, n),
    "stage": np.random.choice(stage_choices, n),
    "age": np.random.randint(20, 51, n)  # 20–50 inclusive
})

# Step 2: add NaN values randomly
nan_ratio = 0.15  # 15% missing

for col in ["sex", "age"]:
    nan_indices = np.random.choice(bigger_df.index, size=int(nan_ratio * n), replace=False)
    bigger_df.loc[nan_indices, col] = np.nan

# Example dataset
small_df = pd.DataFrame({
    "sex": ["M", np.nan, "F"],
    "stage": ["low", "medium", "high"],
    "age": [20, np.nan, 35]
})

In [3]:
processed, num_feats, dummy_feats = DataProcessor.feature_engineering(
    data=small_df,
    data_type="clinical",
    imputer="knn",
    ordinal_cat_columns=["stage"],
    overmissing_threshold=95,
    add_indicators=True,
    verbose=False,
    imputer_params={
        "n_neighbors": 5,
    }
)

In [7]:
processed, num_feats, dummy_feats = DataProcessor.feature_engineering(
    data=bigger_df,
    data_type="clinical",
    imputer="mice",
    ordinal_cat_columns=["stage"],
    overmissing_threshold=95,
    add_indicators=True,
    verbose=True,
    imputer_params= None
)

FEATURE ENGINEERING - OVERMISSING: Removing features with missing > 95%
FEATURE ENGINEERING - METADATA: Classifying feature types for clinical data
  Dummy categorical: 1 | Numerical: 1 | Ordinal: 1
FEATURE ENGINEERING - IMPUTATION: Using MICE
Initialized logger with name MICE Iterations 1 - 20 and 4 levels
1 Dataset 0
 | sex | age
2 Dataset 0
 | sex | age
3 Dataset 0
 | sex | age
4 Dataset 0
 | sex | age
5 Dataset 0
 | sex | age
6 Dataset 0
 | sex | age
7 Dataset 0
 | sex | age
8 Dataset 0
 | sex | age
9 Dataset 0
 | sex | age
10 Dataset 0
 | sex | age
11 Dataset 0
 | sex | age
12 Dataset 0
 | sex | age
13 Dataset 0
 | sex | age
14 Dataset 0
 | sex | age
15 Dataset 0
 | sex | age
16 Dataset 0
 | sex | age
17 Dataset 0
 | sex | age
18 Dataset 0
 | sex | age
19 Dataset 0
 | sex | age
20 Dataset 0
 | sex | age
FEATURE ENGINEERING: Completed


In [8]:
small_df

,sex,stage,age
0,M,low,20.0
1,NaN,medium,NaN
2,F,high,35.0


In [41]:
bigger_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   sex     100 non-null    object 
 1   stage   100 non-null    object 
 2   age     97 non-null     float64
dtypes: float64(1), object(2)
memory usage: 2.5+ KB


In [39]:
processed

,sex,stage,age,missingindicator_age
0,nan,low,26.0,0
1,F,low,39.0,0
2,nan,high,48.0,0
3,F,medium,34.0,0
4,F,medium,30.0,0
...,...,...,...,...
95,M,high,41.0,0
96,F,medium,29.0,0
97,F,medium,23.0,0
98,M,medium,41.0,0


In [5]:
x = DataProcessor.remove_overmissing_features(data=df, threshold=95)
dummy_cat_features, num_features = MetaData.classify_features_types(
                data=x,
                threshold_unique_values=10,
                ordinal_features=["stage"],
            )